# Paper 2 — Few-Shot Transfer Learning for Food Price Forecasting Across Heterogeneous Products

**Research question.** When a product has only K days of price history (a newly listed SKU, or a SKU whose
history is unusable), can knowledge learned from *other* products produce better forecasts than local
methods, and how quickly does the advantage shrink as K grows?

**Design.** Products — not observations — are split: 80 % *seen* products for (meta-)training and 20 %
*unseen* products for evaluation (stratified by volatility; three random product splits). For an unseen
product, every forecast may use only its **last K days**, K in {5, 10, 20, 30, 60}; models are never trained on
unseen products' data, except for the explicit few-shot adaptation step, which uses only the K days before the
test window.

**Methods compared**

| Family | Method |
|---|---|
| Local | Naive; local drift; local SES (alpha selected in-context) |
| Global transfer | Global model trained on full 60-day windows (zero-shot); Global + context-length augmentation (CLA) |
| Adaptation | CLA + per-product fine-tuning of the output layer (L2-SP regularised); Reptile-meta-learned initialisation + fine-tuning |
| **Proposed** | **MetaProto**: CLA + *prototype* conditioning on seen products with similar names in the same category, observed at the same date + Reptile + fine-tuning |
| Reference | Oracle: global model given the full 60-day history (upper bound) |

**Outputs** under `food_price_research/paper2_fewshot/`. Every training run checkpoints each epoch.

## 2. Dataset description

In [ ]:
# ==== 0. CONFIGURATION & GOOGLE DRIVE =====================================
PAPER = "paper2_fewshot"
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("FPR_SMOKE", "0") == "1"   # True -> tiny, fast end-to-end test run
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/food_price_research"
else:
    ROOT = os.environ.get("FPR_ROOT", os.path.abspath("food_price_research"))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'statsforecast', 'scikit-learn'], check=False)

from pathlib import Path
ROOT = Path(ROOT); RAW = ROOT / "raw"; RAW.mkdir(parents=True, exist_ok=True)
PREP_DIR = ROOT / "shared_prep"                      # cleaned panel, shared by all five papers
PAPER_DIR = ROOT / PAPER
FIG_DIR, TAB_DIR, RES_DIR, CKPT_DIR = [PAPER_DIR / s for s in ("figures", "tables", "results", "checkpoints")]
for d in (PREP_DIR, FIG_DIR, TAB_DIR, RES_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

cands = sorted(RAW.glob("[dD]ataset-2024.*"))
if not cands and IN_COLAB:
    from google.colab import files
    print("Upload dataset-2024.csv (or Dataset-2024.xlsx) - it is saved to", RAW)
    for k, v in files.upload().items():
        (RAW / k).write_bytes(v)
    cands = sorted(RAW.glob("[dD]ataset-2024.*"))
assert cands, f"Put dataset-2024.csv in {RAW}"
DATA_PATH = cands[0]
print("paper:", PAPER, "| data:", DATA_PATH, "| smoke:", SMOKE)

## 1. Shared data preparation (identical in all five notebooks)

The raw panel is a wide table (`Name | 2024-01-02 | ... | 2025-02-05`): 5,976 SKUs x 401 days, already
complete. Quality control, all estimated on training data only:

* **Scraper outages.** On 56 days not a single SKU changes price (the median day has ~80 changes): the scraper
  did not run and prices were carried forward. These cells are *imputed*, so they are excluded from every loss,
  metric target and graph statistic. The first day after an outage (a *gap-end* day) accumulates several days of
  changes and is excluded from co-movement statistics.
* **Spike-and-revert repair.** A jump of more than x1.5 that returns to within 5 % of the previous level within
  7 days is treated as a scrape error (wrong variant shown) and replaced by the previous level.
* **Variant breaks.** A persistent level shift of more than x3 within the training window is almost surely a
  re-used product slot; those SKUs are excluded.
* **Platform batch-update days.** Days whose number of simultaneous changes exceeds median + 5 MAD. Batch
  updates create spurious co-movement, so they are excluded from correlation-type statistics.
* **Category tagger.** Priority-ordered keyword rules on product names; non-food SKUs are kept as a control
  group and excluded from headline results.
* **Activity tiers** (on the earliest training window, so no look-ahead): T0 food SKUs that never change,
  T1 1-9 changes, T2 >= 10 changes. Headline results use the *active* universe T1 u T2.
* **Rolling-origin evaluation.** Three folds, each with a 30-day validation and a 30-day test window
  (test windows: 2024-11-08..12-07, 12-08..01-06, 2025-01-07..02-05). Models never see validation or test
  targets during fitting; forecasts at each origin use only data up to that origin.
* **Targets.** 100 x log(P[t+h] / P[t]) for h = 1..30; metrics are computed in price space at h in {1,3,7,14,30}.
  Because most prices are sticky we also report **CC-sMAPE** (sMAPE only where the price actually changes) and
  **change AUPRC**, alongside MAE, RMSE, MAPE, sMAPE, WAPE and MASE.

In [ ]:
# ============================================================================
# SECTION 1 — SHARED DATA PREPARATION (identical, deterministic in every notebook)
# ============================================================================
import re, json, time, math, pickle, hashlib, warnings, itertools
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1.1  Category tagger (rule-based, from product names; priority-ordered)
# ---------------------------------------------------------------------------
# Non-food rules are checked FIRST so that e.g. "hair oil" never becomes edible
# oil and "baby shampoo" never becomes baby food.
CATEGORY_RULES = [
    # ---- non-food -------------------------------------------------------
    ("NF: Personal Care", r"shampoo|conditioner|face ?wash|facewash|facial|soap|lotion|deodorant|deo |roll on|perfume|body spray|"
                          r"toothpaste|toothbrush|tooth ?brush|mouthwash|hair oil|hair color|hair colour|hair removal|hair treatment|"
                          r"serum|sunscreen|sun protect|talcum|cream(?!.*(biscuit|wafer|cake|cheese|ice cream|milk|roll|sandwich))|"
                          r"vaseline|petroleum jelly|nose strip|shaving|razor|blade|sanitary|napkin|pad[s]? |tampon|condom|lip ?balm|"
                          r"lipstick|nail|kajal|mascara|glycerine|cleansing|scrub|makeup|cosmetic|beauty|skin|aloe ?vera gel|"
                          r"hand ?wash|body wash|shower gel|cotton bud|wipes|essential oil|facial oil|massage oil|coconut oil hair|"
                          r"boroplus|moisturi|toner|face mask|mask|blackhead|dandruff|gatsby|gloss|hair gel|lip colou?r|hair|shave|sanitizer|baby wash|baby bath|\blip"),
    ("NF: Baby Care",     r"diaper|nappy|pant [smlx]|belt \d|feeding bottle|bottles cleanser|nipple|baby pillow|nima|teether|"
                          r"baby lotion|baby oil|baby soap|baby shampoo|baby powder|baby wipes|pacifier|romper|cot sheet|pants .*kg"),
    ("NF: Household",     r"clean(er|ing)|detergent|dish ?wash|washing|bleach|disinfect|toilet|harpic|lizol|air freshener|"
                          r"freshener|aerosol|mosquito|insect|coil|killer|repellent|tissue|napkin|foil|cling film|garbage|"
                          r"broom|mop|brush|sponge|scrubber|battery|bulb|light|candle|matchbox|match box|lighter|polish|"
                          r"glass cleaner|floor|mat |mat$|pillow|towel|bucket|container|organizer|box$|jar |bottle(?!.*(juice|drink|water|milk|sauce|oil))|"
                          r"plate|bowl|spoon|glass |mug|cup(?! noodles)|tray|chopping board|knife|rolling pin|leveler|grill|net$|pan |pot |kettle|"
                          r"flask|tiffin|lunch box|hanger|umbrella|finish household|powerball|fabric|softener|surf excel|rin |wheel |"
                          r"cat food|dog food|pet |\bcat\b|kitten|\bdog\b|smartheart|pro diet|whiskas|pedigree|kitty|wrap|mixer|apron|strainer|duster|dish cover|holder|litter|"
                          r"spray|cockroach|\bhit\b|diesel|\bmobil\b|lubricant|\bcover\b|sheet"),
    ("NF: Health",        r"tablet|capsule|syrup(?!.*(chocolate|strawberry))|vitamin|antiseptic|dettol liquid|savlon|bandage|"
                          r"plaster|thermometer|oral saline|ors|pain relief|balm|inhaler|medicine|n95|surgical|glucose(?! biscuit)|"
                          r"test kit|pregnancy"),
    ("NF: Stationery & Misc", r"pencil|pen |pen$|ball ?pen|marker|highlighter|eraser|sharpener|notebook|note ?book|file |folder|"
                          r"stapler|punch|scissor|cutter|glue|tape|calculator|compass|geometry|ruler|a4|paper|envelope|sticker|"
                          r"flag|toy|puzzle|ball |badminton|shuttlecock|cricket|football|game|doll|remote|charger|cable|"
                          r"earphone|headphone|usb|mobile|phone|fusion set|moistener|clip|board|diary|book|colour pencil|crayon|"
                          r"gift|decoration|balloon|travel|bag|wallet|\bcard\b|\bbear\b|\bcar\b|instrument|acrylic|glitter|sash|tasbih|"
                          r"\bbat\b|lace|drawing|flower"),
    # ---- food ---------------------------------------------------------------
    ("Baby Food",         r"formula|infant|baby food|cerelac|nan optipro|lactogen|biomil|growing up milk|\(0-6|\(6-12|\(1-3|"
                          r"nido 1|nido 3|cereal.*(tin|baby)|mother's smile"),
    ("Beverages",         r"juice|drink|beverage|electrolyte|soda|cola|7 ?up|sprite|fanta|pepsi|coca|mojo|speed|energy|water|lassi|milk shake|"
                          r"shake|tang|horlicks|boost|ovaltine|bournvita|malt|squash|syrup|can$|smoothie|mocktail|aloe drink|frooto|frutika"),
    ("Tea & Coffee",      r"\btea\b|tea bag|green tea|coffee|nescafe|infusion|chai"),
    ("Dairy & Eggs",      r"milk|cheese|butter(?!.*(cookie|biscuit|peanut))|yogurt|yoghurt|\bdoi\b|curd|cream cheese|paneer|"
                          r"ghee|egg|dim\b|dairy|uht|evaporated|condensed|ice cream|kulfi"),
    ("Edible Oil",        r"soybean oil|soyabean|mustard oil|sunflower oil|palm oil|rice bran|olive oil|canola|"
                          r"vegetable oil|cooking oil|sesame oil|\boil\b"),
    ("Fish & Seafood",    r"fish|ilish|hilsha|rui|rohu|katla|pangas|tilapia|shing|magur|koi|pabda|chingri|shrimp|prawn|"
                          r"crab|squid|baila|boal|shol|taki|poa|rupchanda|pomfret|loitta|kachki|mola|tengra|shutki|dried fish"),
    ("Meat & Poultry",    r"beef|mutton|goat|lamb|chicken|broiler|deshi murgi|murgi|duck(?! eggs)|pigeon|quail|meat|"
                          r"liver|gizzard|keema|mince|bone in|boneless|drumstick|breast|wings|sausage|salami|nugget|patty|kebab"),
    ("Frozen & Ready",     r"frozen|ready to cook|ready mix|paratha|puri|samosa|singara|spring roll|haleem mix|biryani mix|"
                          r"instant|french fries|burger|pizza(?! sauce)"),
    ("Rice & Grains",     r"\brice\b|chinigura|basmati|miniket|nazirshail|atop|kalijira|polao|poha|chira|muri|khoi|oats|"
                          r"cornflakes|corn flakes|cereal|muesli|barley|quinoa|semai|shemai|laccha|lachcha|vermicelli|wheat"),
    ("Flour & Bakery",    r"atta|maida|flour|suji|semolina|besan|bread|bun|cake|rusk|toast|bakorkhani|bakery|dough|yeast|"
                          r"baking|pastry|croissant|donut|ruti|chapati|roti"),
    ("Lentils & Pulses",  r"\bdal\b|lentil|masur|moong|mung|mashkalai|chola|chick ?pea|kabuli|anchor dal|khesari|"
                          r"motor|peas|beans?\b|rajma"),
    ("Spices & Condiments", r"masala|spice|powder|chili|chilli|turmeric|holud|coriander|cumin|zeera|jeera|cardamom|elach|"
                          r"cinnamon|daruchini|clove|lobongo|black pepper|golmorich|bay leaf|tejpata|nutmeg|jayfal|mace|"
                          r"fenugreek|methi|panch phoron|mix$|salt|vinegar|flakes|seasoning|stock|soup|bouillon|gelatine|"
                          r"food colou?r|essence|baking soda|kewra|rose water|seed|herbs|china grass|mix\s*$"),
    ("Sugar & Sweeteners", r"sugar|chini|\bgur\b|jaggery|honey|mishri|sweetener|candy|chocolate|toffee|lollipop|jelly|"
                          r"marshmallow|gum|mint"),
    ("Snacks & Biscuits", r"biscuit|cookie|cracker|wafer|chips|chanachur|namkeen|dalmoth|bhujia|nuts|peanut(?! butter)|"
                          r"cashew|kaju|almond|pista|raisin|kismis|dates|khejur|popcorn|puffs|roll$|snack|bar |bar$|"
                          r"chanachur|jhal muri|sticks|crisps|pringles|kurkure|lays|badam"),
    ("Sauces & Spreads",  r"sauce|ketchup|mayonnaise|mayo|\bjam\b|jelly|spread|pickle|achar|chutney|peanut butter|nutella|"
                          r"dressing|paste|puree|olive(?! oil)"),
    ("Noodles & Pasta",   r"noodle|ramen|pasta|macaroni|spaghetti|fusilli|penne|maggi|cup noodles"),
    ("Fruits",            r"mango|aam|banana|kola\b|apple|orange|malta|grape|guava|peyara|papaya|pape|pineapple|anaros|"
                          r"watermelon|tormuj|lemon|lebu|lime|pomegranate|anar|litchi|lichu|jackfruit|kathal|coconut|"
                          r"narikel|dates|berry|kiwi|pear|plum|prune|wood apple|kodbel|amra|jambura|sofeda|fruit"),
    ("Vegetables",        r"potato|alu|aloo|onion|peyaj|piyaj|garlic|roshun|ginger|ada\b|tomato|brinjal|begun|cucumber|"
                          r"shosha|pumpkin|kumra|gourd|lau|korola|potol|dherosh|okra|cabbage|cauliflower|kopi|carrot|gajor|"
                          r"radish|mula|spinach|shak|sak\b|lettuce|capsicum|green chili|kacha morich|beans|borboti|"
                          r"mushroom|coriander leaves|dhonia pata|mint leaves|pudina|vegetable|sobji|corn|bhutta|"
                          r"betel|shupari|paan|chal kumra|kochu|arum|shim|motorshuti|lemongrass|drumstick|broccoli|\bpui\b|microgreen|sheem"),
]
NONFOOD_PREFIX = "NF: "
_COMPILED = [(c, re.compile(p, re.I)) for c, p in CATEGORY_RULES]

def fix_mojibake(s: str) -> str:
    """Repair UTF-8 read as cp1252 (e.g. 'Â±50 gm' -> '±50 gm')."""
    try:
        t = s.encode("cp1252").decode("utf-8")
        return t
    except Exception:
        return s.replace("Â±", "±").replace("Â ", " ").replace("â€™", "'")

def tag_category(name: str) -> str:
    n = " " + name.lower().strip() + " "
    for cat, rx in _COMPILED:
        if rx.search(n):
            return cat
    return "Unclassified"

# ---------------------------------------------------------------------------
# 1.2  Load the wide panel  (Name | date1 | date2 | ...)
# ---------------------------------------------------------------------------
def load_panel(path):
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xls"):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path, encoding="utf-8-sig")
    df = df.rename(columns={df.columns[0]: "Name"})
    date_cols = [c for c in df.columns[1:]]
    dates = pd.to_datetime([str(c)[:10] for c in date_cols])
    P = df[date_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    names = df["Name"].astype(str).map(fix_mojibake).str.strip().tolist()
    return names, dates, P

# ---------------------------------------------------------------------------
# 1.3  Data-quality: interpolated-window mask, spike-and-revert repair,
#      persistent variant breaks, platform batch-update days
# ---------------------------------------------------------------------------
def detect_carried_forward_days(P, max_changes=2):
    """Scraper-outage days. When the scraper did not run, the panel carries the
    previous day's prices forward, so NOT A SINGLE product changes price that
    day (the median day has ~80 changes). These cells are imputed, not observed:
    they are excluded from every loss/metric target and from graph statistics.
    The first observed day after an outage accumulates several days of changes
    (a 'gap-end' day) and is flagged separately."""
    ch = np.concatenate([[10**9], (np.diff(P, axis=1) != 0).sum(axis=0)])
    cf = ch <= max_changes
    gap_end = np.zeros_like(cf)
    gap_end[1:] = cf[:-1] & ~cf[1:]
    return cf, gap_end

def repair_spikes(P, tau=np.log(1.5), max_len=7, tol=0.05):
    """Spike-and-revert repair: a jump |dlog| > tau that returns to within `tol`
    of the pre-jump level within `max_len` days is treated as a scrape error
    (wrong variant / pack size shown for a few days) and replaced by the
    pre-jump level. Returns cleaned matrix + per-series repair counts."""
    L = np.log(P)
    Q = P.copy(); n_rep = np.zeros(P.shape[0], dtype=int)
    T = P.shape[1]
    for i in range(P.shape[0]):
        x = L[i]
        d = np.diff(x)
        cand = np.where(np.abs(d) > tau)[0]
        t_skip = -1
        for j in cand:
            if j <= t_skip: continue
            base = x[j]
            for k in range(j + 1, min(j + 1 + max_len, T - 1)):
                if abs(x[k + 1] - base) <= tol:
                    Q[i, j + 1: k + 1] = P[i, j]
                    n_rep[i] += 1; t_skip = k
                    break
    return Q, n_rep

def detect_variant_breaks(P, t_end, thr=np.log(3.0)):
    """Persistent level shifts > x3 inside the training window: almost surely a
    re-used product slot (different pack/variant), not a price change."""
    d = np.abs(np.diff(np.log(P[:, : t_end + 1]), axis=1))
    return (d > thr).any(axis=1)

def detect_batch_days(P, t_end, k_mad=5.0):
    """Days where the number of simultaneous price changes is an outlier
    (median + k*MAD, threshold estimated on training days only)."""
    ch = (np.diff(P, axis=1) != 0).sum(axis=0)          # changes on day t (t>=1)
    ch = np.concatenate([[0], ch])
    tr = ch[1: t_end + 1]
    med = np.median(tr); mad = np.median(np.abs(tr - med)) + 1e-9
    thr = med + k_mad * 1.4826 * mad
    return ch > thr, ch, thr

# ---------------------------------------------------------------------------
# 1.4  Rolling-origin splits (3 folds, 30-day val + 30-day test each)
# ---------------------------------------------------------------------------
def make_folds(T, n_folds=3, test_len=30, val_len=30):
    folds = []
    for k in range(n_folds):
        test_end = T - 1 - test_len * (n_folds - 1 - k)
        test_start = test_end - test_len + 1
        val_end = test_start - 1; val_start = val_end - val_len + 1
        folds.append(dict(fold=k, train_end=val_start - 1, val_start=val_start, val_end=val_end,
                          test_start=test_start, test_end=test_end))
    return folds

# ---------------------------------------------------------------------------
# 1.5  Master prep (cached to Drive; identical in all five notebooks)
# ---------------------------------------------------------------------------
PREP_VERSION = "v1.1"

def build_prep(data_path, cache_dir, clean_spikes=True, force=False):
    cache_dir = Path(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)
    f = cache_dir / f"prep_{PREP_VERSION}.pkl"
    if f.exists() and not force:
        with open(f, "rb") as fh:
            prep = pickle.load(fh)
        print(f"[prep] loaded cached panel from {f}")
        return prep
    t0 = time.time()
    names, dates, P_raw = load_panel(data_path)
    N, T = P_raw.shape
    assert np.isfinite(P_raw).all() and (P_raw > 0).all(), "panel must be complete & positive"
    folds = make_folds(T)
    tier_end = folds[0]["train_end"]                    # leak-free: earliest training window
    cf_day, gap_end = detect_carried_forward_days(P_raw)
    P_clean, n_rep = repair_spikes(P_raw) if clean_spikes else (P_raw.copy(), np.zeros(N, int))
    variant_break = detect_variant_breaks(P_clean, tier_end)
    batch_day, daily_changes, batch_thr = detect_batch_days(P_clean, tier_end)
    cats = [tag_category(n) for n in names]
    is_food = np.array([not c.startswith(NONFOOD_PREFIX) and c != "Unclassified" for c in cats])
    # activity statistics on the tier window only (no look-ahead)
    dlog_tr = np.diff(np.log(P_clean[:, : tier_end + 1]), axis=1)
    n_changes = (dlog_tr != 0).sum(axis=1)
    vol = dlog_tr.std(axis=1)
    usable = is_food & ~variant_break
    tier = np.full(N, "T0", dtype=object)
    tier[usable & (n_changes >= 1)] = "T1"
    tier[usable & (n_changes >= 10)] = "T2"
    # volatility terciles among T1 (training-window std of log-returns)
    vol_bucket = np.full(N, "none", dtype=object)
    m = usable & (n_changes >= 1)
    q1, q2 = np.quantile(vol[m], [1 / 3, 2 / 3])
    vol_bucket[m & (vol <= q1)] = "low"
    vol_bucket[m & (vol > q1) & (vol <= q2)] = "medium"
    vol_bucket[m & (vol > q2)] = "high"
    meta = pd.DataFrame(dict(name=names, category=cats, is_food=is_food, variant_break=variant_break,
                             n_changes_train=n_changes, vol_train=vol, tier=tier, vol_bucket=vol_bucket,
                             spike_repairs=n_rep, mean_price_train=P_clean[:, : tier_end + 1].mean(axis=1)))
    prep = dict(names=names, dates=dates, P_raw=P_raw, P=P_clean, logP=np.log(P_clean), meta=meta,
                folds=folds, tier_end=tier_end, cf_day=cf_day, gap_end=gap_end,
                batch_day=batch_day, daily_changes=daily_changes, batch_thr=batch_thr,
                version=PREP_VERSION, built=time.strftime("%Y-%m-%d %H:%M"))
    with open(f, "wb") as fh:
        pickle.dump(prep, fh)
    print(f"[prep] built in {time.time()-t0:.1f}s and cached to {f}")
    return prep

def subset_idx(prep, tiers=("T1", "T2"), food_only=True, smoke_n=None, seed=0):
    meta = prep["meta"]
    m = meta.tier.isin(tiers).to_numpy()
    if food_only: m &= meta.is_food.to_numpy()
    idx = np.where(m)[0]
    if smoke_n is not None and len(idx) > smoke_n:
        rng = np.random.default_rng(seed); idx = np.sort(rng.choice(idx, smoke_n, replace=False))
    return idx

In [ ]:
# ============================================================================
# SECTION 1b — SHARED UTILITIES: figures, tables, checkpoints, windows, metrics
# ============================================================================
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats as sps

# ---------------------------------------------------------------------------
# Figure style (journal): 8-pt sans, thin marks, recessive grid, fixed palette
# ---------------------------------------------------------------------------
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GREY = "#8a8985"
COL1, COL2 = 3.5, 7.2                       # single / double column widths (inches)
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.6, "axes.edgecolor": "#52514e", "axes.grid": True, "grid.color": "#e4e3df",
    "grid.linewidth": 0.5, "lines.linewidth": 1.4, "lines.markersize": 4, "legend.frameon": False,
    "axes.prop_cycle": matplotlib.cycler(color=PALETTE), "pdf.fonttype": 42, "ps.fonttype": 42,
})

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{name}.{ext}")
    print(f"[fig] {name} -> {FIG_DIR}")
    plt.show(); plt.close(fig)

# ---------------------------------------------------------------------------
# Tables: CSV + LaTeX (booktabs), no jinja2 dependency; optional bolding of best
# ---------------------------------------------------------------------------
def _fmt(v, nd):
    if isinstance(v, (float, np.floating)):
        if not np.isfinite(v): return "--"
        return f"{v:.{nd}f}"
    return str(v).replace("_", r"\_").replace("&", r"\&").replace("%", r"\%")

def save_table(df, name, caption="", label=None, nd=3, bold_min=None, bold_max=None, index=False, bold_row_min=None):
    """Write <name>.csv and <name>.tex (booktabs). bold_min/bold_max: list of
    columns in which the best value is set in bold."""
    df = df.reset_index() if index else df.copy()
    df.to_csv(TAB_DIR / f"{name}.csv", index=False)
    best = {}
    for c in (bold_min or []):
        if c in df: best[c] = pd.to_numeric(df[c], errors="coerce").min()
    for c in (bold_max or []):
        if c in df: best[c] = pd.to_numeric(df[c], errors="coerce").max()
    cols = list(df.columns)
    align = "".join("l" if not np.issubdtype(df[c].dtype, np.number) else "r" for c in cols)
    lines = [r"\begin{table}[t]", r"\centering", r"\small"]
    if caption: lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label or 'tab:' + name}}}")
    lines += [rf"\begin{{tabular}}{{{align}}}", r"\toprule",
              " & ".join(_fmt(c, nd) for c in cols) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        cells = []
        rmin = pd.to_numeric(r[bold_row_min], errors="coerce").min() if bold_row_min else None
        for c in cols:
            s = _fmt(r[c], nd)
            if np.issubdtype(df[c].dtype, np.integer): s = str(int(r[c]))
            if bold_row_min and c in bold_row_min and np.isfinite(r[c]) and np.isclose(r[c], rmin):
                s = rf"\textbf{{{s}}}"
            elif c in best and isinstance(r[c], (float, np.floating, int, np.integer)) and np.isfinite(r[c]) \
               and np.isclose(r[c], best[c]):
                s = rf"\textbf{{{s}}}"
            cells.append(s)
        lines.append(" & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (TAB_DIR / f"{name}.tex").write_text("\n".join(lines))
    print(f"[table] {name} -> {TAB_DIR}")
    try:
        display(df)
    except NameError:
        print(df.to_string())
    return df

# ---------------------------------------------------------------------------
# Result caching & checkpoints (everything resumable after a Colab disconnect)
# ---------------------------------------------------------------------------
def cached(name, fn, force=False):
    f = RES_DIR / f"{name}.pkl"
    if f.exists() and not force:
        with open(f, "rb") as fh: return pickle.load(fh)
    out = fn()
    with open(f, "wb") as fh: pickle.dump(out, fh)
    return out

def set_seed(s):
    import random
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass

# ---------------------------------------------------------------------------
# Feature tensor & rolling windows
#   channels: 0 = 100*log(P_t'/P_origin)   (filled at gather time)
#             1 = price-changed indicator  2 = observed (not carried-forward)
#             3,4 = day-of-week sin/cos     5 = log1p(days since last change)/5
#   targets : 100*log(P_{o+h}/P_o), h = 1..H, clipped to [-100, 100]
# ---------------------------------------------------------------------------
H_MAX, H_EVAL = 30, [1, 3, 7, 14, 30]
N_CH = 6

def build_features(prep, idx):
    lp = prep["logP"][idx]
    n, T = lp.shape
    chg = np.zeros_like(lp); chg[:, 1:] = (np.diff(lp, axis=1) != 0)
    obs = np.broadcast_to((~prep["cf_day"]).astype(np.float64), (n, T))
    dow = prep["dates"].dayofweek.to_numpy()
    s, c = np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7)
    dsc = np.zeros_like(lp)
    for t in range(1, T):
        dsc[:, t] = np.where(chg[:, t] > 0, 0, dsc[:, t - 1] + 1)
    F = np.stack([lp, chg, obs, np.broadcast_to(s, (n, T)), np.broadcast_to(c, (n, T)),
                  np.log1p(dsc) / 5], axis=-1).astype(np.float32)
    return F                                                   # (n, T, C); channel 0 = raw log-price

def origins_for(fold, part, L, H=H_MAX):
    """Forecast origins o (inclusive) for 'train' | 'val' | 'test'. Targets at
    o+1..o+H are masked beyond the segment end (no peeking)."""
    if part == "train":
        return np.arange(L - 1, fold["train_end"]), fold["train_end"]
    if part == "val":
        return np.arange(fold["val_start"] - 1, fold["val_end"]), fold["val_end"]
    return np.arange(fold["test_start"] - 1, fold["test_end"]), fold["test_end"]

def gather_windows(F, idx, origins, L, H, seg_end, cf_day, rows=None):
    """Vectorised, ORIGIN-MAJOR gather for products `rows` (local indices into F,
    default all) at forecast origins `origins`.
    Returns X (k, n, L, C), Y (k, n, H), M (k, n, H) and meta (k, n, 2) holding
    (global product id, origin day). Channel 0 of X is re-based to the origin."""
    rows = np.arange(F.shape[0]) if rows is None else np.asarray(rows)
    oo, rr = np.meshgrid(origins, rows, indexing="ij")                    # (k, n)
    tt = oo[..., None] + np.arange(-L + 1, 1)                              # (k, n, L)
    X = F[rr[..., None], np.maximum(tt, 0)].copy()
    base = F[rr, oo, 0]
    X[..., 0] = 100.0 * (X[..., 0] - base[..., None])
    hh = oo[..., None] + np.arange(1, H + 1)                               # (k, n, H)
    valid = hh <= seg_end
    hc = np.minimum(hh, F.shape[1] - 1)
    M = valid & ~cf_day[hc]
    Y = 100.0 * (F[rr[..., None], hc, 0] - base[..., None])
    Y = np.clip(np.where(M, Y, 0.0), -100, 100).astype(np.float32)
    meta = np.stack([np.asarray(idx)[rr], oo], -1)
    return X.astype(np.float32), Y, M, meta

# ---------------------------------------------------------------------------
# Evaluation in PRICE space. Predictions are 100*log-ratio vs the origin price.
# ---------------------------------------------------------------------------
def price_scale(prep, idx, fold):
    """MASE scale: mean |P_t - P_{t-1}| on the fold's training window."""
    P = prep["P"][idx, : fold["train_end"] + 1]
    return np.maximum(np.abs(np.diff(P, axis=1)).mean(axis=1), 1e-6)

def evaluate(pred, Y, M, meta, prep, fold, horizons=H_EVAL):
    """pred, Y, M: (..., H) arrays (any leading shape); meta (..., 2).
    Returns (summary per horizon, {h: per-product mean sAPE})."""
    H = Y.shape[-1]
    pred, Y, M, meta = pred.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H), meta.reshape(-1, 2)
    P0 = prep["P"][meta[:, 0], meta[:, 1]]
    gids = np.unique(meta[:, 0])
    scale = pd.Series(price_scale(prep, gids, fold), index=gids)
    sc = scale.loc[meta[:, 0]].to_numpy()
    rows = []
    per_prod = {}
    for h in horizons:
        m = M[:, h - 1]
        if m.sum() == 0: continue
        y = P0[m] * np.exp(Y[m, h - 1] / 100); yh = P0[m] * np.exp(pred[m, h - 1] / 100)
        e = yh - y; ae = np.abs(e)
        sape = 200 * ae / (np.abs(y) + np.abs(yh))
        chg = np.abs(Y[m, h - 1]) > 1e-9
        prod = meta[m, 0]
        df = pd.DataFrame(dict(prod=prod, ae=ae, sape=sape, se=ae**2, ape=100 * ae / y, sc_ae=ae / sc[m], y=y))
        g = df.groupby("prod")
        mase_p = g.sc_ae.mean()
        rows.append(dict(h=h, n=int(m.sum()), MAE=ae.mean(), RMSE=np.sqrt((ae**2).mean()),
                         MAPE=df.ape.mean(), sMAPE=sape.mean(), WAPE=100 * ae.sum() / y.sum(),
                         MASE=mase_p.mean(), MASE_med=mase_p.median(),
                         CC_sMAPE=sape[chg].mean() if chg.any() else np.nan,
                         change_rate=chg.mean()))
        per_prod[h] = g.sape.mean()
    return pd.DataFrame(rows), per_prod

def change_auprc(score, Y, M, h):
    """AUPRC for 'will the price differ from today's price at horizon h?'."""
    from sklearn.metrics import average_precision_score
    H = Y.shape[-1]; score, Y, M = score.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H)
    m = M[:, h - 1]; lab = np.abs(Y[m, h - 1]) > 1e-9
    if lab.sum() == 0 or lab.all(): return np.nan
    return average_precision_score(lab, score[m, h - 1])

# ---------------------------------------------------------------------------
# Statistical tests
# ---------------------------------------------------------------------------
def dm_test(l1, l2, h=1):
    """Diebold-Mariano with Harvey-Leybourne-Newbold small-sample correction.
    l1, l2: loss series (e.g. cross-sectional mean sAPE per origin). Negative
    statistic => model 1 more accurate."""
    d = np.asarray(l1) - np.asarray(l2); d = d[np.isfinite(d)]
    T = len(d)
    if T < 5: return np.nan, np.nan
    dbar = d.mean(); g = [np.mean((d[k:] - dbar) * (d[: T - k] - dbar)) for k in range(min(h, T - 1))]
    v = (g[0] + 2 * sum(g[1:])) / T
    if v <= 0: return np.nan, np.nan
    stat = dbar / np.sqrt(v) * np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    return stat, 2 * sps.t.sf(abs(stat), df=T - 1)

def wilcoxon_paired(a, b):
    a, b = pd.Series(a), pd.Series(b); j = a.index.intersection(b.index)
    d = (a[j] - b[j]).to_numpy()
    if np.allclose(d, 0): return np.nan
    return sps.wilcoxon(d, zero_method="zsplit").pvalue

def bootstrap_ci(d, B=2000, seed=0, alpha=0.05):
    d = np.asarray(d); d = d[np.isfinite(d)]
    rng = np.random.default_rng(seed)
    bs = rng.choice(d, (B, len(d)), replace=True).mean(axis=1)
    return d.mean(), np.quantile(bs, alpha / 2), np.quantile(bs, 1 - alpha / 2)

def holm(pvals):
    p = np.asarray(pvals, dtype=float); out = np.full_like(p, np.nan)
    ok = np.isfinite(p); idx = np.argsort(p[ok]); m = ok.sum()
    adj = np.maximum.accumulate((m - np.arange(m)) * p[ok][idx]); adj = np.minimum(adj, 1)
    tmp = np.empty(m); tmp[idx] = adj; out[ok] = tmp
    return out

def per_origin_loss(pred, Y, M, meta, prep, h):
    """Cross-sectional mean sAPE per origin at horizon h (DM-test input)."""
    H = Y.shape[-1]
    pred, Y, M, meta = pred.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H), meta.reshape(-1, 2)
    m = M[:, h - 1]
    P0 = prep["P"][meta[m, 0], meta[m, 1]]
    y = P0 * np.exp(Y[m, h - 1] / 100); yh = P0 * np.exp(pred[m, h - 1] / 100)
    s = pd.Series(200 * np.abs(yh - y) / (y + yh)).groupby(meta[m, 1]).mean()
    return s

# ---------------------------------------------------------------------------
# Dataset description (figure + table used in the Data section of every paper)
# ---------------------------------------------------------------------------
def dataset_overview(prep):
    meta, dates = prep["meta"], prep["dates"]
    rows = [("Products (SKUs)", len(meta)), ("Days", len(dates)),
            ("Period", f"{dates[0].date()} to {dates[-1].date()}"),
            ("Food SKUs (tagged)", int(meta.is_food.sum())),
            ("Excluded: variant breaks", int(meta.variant_break.sum())),
            ("Scraper-outage days (carried forward)", int(prep["cf_day"].sum())),
            ("Platform batch-update days", int(prep["batch_day"].sum())),
            ("Spike-and-revert repairs", int(meta.spike_repairs.sum())),
            ("T0 (food, never changes in train)", int(((meta.tier == "T0") & meta.is_food).sum())),
            ("T1 (food, 1-9 changes)", int((meta.tier == "T1").sum())),
            ("T2 (food, >=10 changes)", int((meta.tier == "T2").sum()))]
    save_table(pd.DataFrame(rows, columns=["Item", "Value"]).astype({"Value": str}), "data_summary",
               "Dataset summary after quality control.")
    fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.2))
    ch = prep["daily_changes"]
    ax[0].bar(dates, ch, width=1.0, color=PALETTE[0], lw=0)
    ax[0].bar(dates[prep["batch_day"]], ch[prep["batch_day"]], width=2.5, color=PALETTE[1], lw=0, label="batch update")
    ax[0].scatter(dates[prep["cf_day"]], np.zeros(prep["cf_day"].sum()), marker="|", s=30, color=GREY, label="outage (0 changes)")
    ax[0].set_title("(a) Daily number of price changes"); ax[0].legend(loc="upper left")
    ax[0].tick_params(axis="x", rotation=30)
    nc = meta.loc[meta.is_food, "n_changes_train"]
    ax[1].hist(np.log10(nc + 1), bins=30, color=PALETTE[0], edgecolor="white", lw=0.4)
    ax[1].set_xlabel("log$_{10}$(1 + price changes in training window)"); ax[1].set_ylabel("Food SKUs")
    ax[1].set_title("(b) Price stickiness")
    act = meta[meta.tier.isin(["T1", "T2"])]
    order = act.category.value_counts()
    ax[2].barh(order.index[::-1], order.values[::-1], color=PALETTE[0])
    ax[2].set_title("(c) Active food SKUs by category"); ax[2].tick_params(axis="y", labelsize=6)
    fig.tight_layout(); savefig(fig, "fig_data_overview")

# ---------------------------------------------------------------------------
# Fold data bundle: windows for train / val / test in one call (cached in RAM)
# ---------------------------------------------------------------------------
def fold_data(prep, F, idx, fold, L, H=H_MAX, parts=("train", "val", "test"), train_stride=1):
    out = {}
    for part in parts:
        O, seg_end = origins_for(fold, part, L, H)
        if part == "train": O = O[::train_stride]
        out[part] = gather_windows(F, idx, O, L, H, seg_end, prep["cf_day"])
    return out

def naive_family(X, H=H_MAX):
    """Naive (0), seasonal-naive(7) and moving-average(7) forecasts from windows.
    X channel 0 is 100*log(P/P_origin) so the naive forecast is identically 0."""
    k = X[..., 0]
    naive = np.zeros(k.shape[:-1] + (H,), np.float32)
    L = k.shape[-1]
    snaive = np.stack([k[..., L - 7 + ((h - 1) % 7)] for h in range(1, H + 1)], -1)
    ma = np.repeat(k[..., -7:].mean(-1, keepdims=True), H, -1)
    return {"Naive": naive, "SNaive-7": snaive.astype(np.float32), "MA-7": ma.astype(np.float32)}

def results_frame(res_dict):
    """{model: summary-df} -> long frame with a 'model' column."""
    return pd.concat([d.assign(model=m) for m, d in res_dict.items()], ignore_index=True)

In [ ]:
# ============================================================================
# SECTION 1c — SHARED PYTORCH UTILITIES: resumable trainer + building blocks
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as Fnn
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print("device:", DEVICE)

def masked_huber(pred, y, m, delta=1.0):
    m = m.float()
    l = Fnn.huber_loss(pred, y, reduction="none", delta=delta)
    return (l * m).sum() / m.sum().clamp(min=1.0)

def iterate_flat(X, Y, M, bs, shuffle, rng=None, extra=None):
    """Mini-batches over flattened (origin, product) windows. X: (k,n,L,C).
    extra: dict of (k,n,...) arrays gathered alongside (e.g. category ids)."""
    k, n = X.shape[:2]
    N = k * n
    order = (rng or np.random.default_rng(0)).permutation(N) if shuffle else np.arange(N)
    Xf, Yf, Mf = X.reshape(N, *X.shape[2:]), Y.reshape(N, -1), M.reshape(N, -1)
    ex = {kk: v.reshape(N, *v.shape[2:]) for kk, v in (extra or {}).items()}
    for s in range(0, N, bs):
        j = order[s: s + bs]
        b = dict(x=torch.from_numpy(Xf[j]).to(DEVICE), y=torch.from_numpy(Yf[j]).to(DEVICE),
                 m=torch.from_numpy(Mf[j]).to(DEVICE))
        for kk, v in ex.items(): b[kk] = torch.from_numpy(v[j]).to(DEVICE)
        yield b

def iterate_snap(X, Y, M, bs, shuffle, rng=None, extra=None):
    """Mini-batches of whole cross-sections (all products at an origin) for
    graph / cross-product models. X: (k, n, L, C) -> batch x: (b, n, L, C)."""
    k = X.shape[0]
    order = (rng or np.random.default_rng(0)).permutation(k) if shuffle else np.arange(k)
    for s in range(0, k, bs):
        j = order[s: s + bs]
        b = dict(x=torch.from_numpy(X[j]).to(DEVICE), y=torch.from_numpy(Y[j]).to(DEVICE),
                 m=torch.from_numpy(M[j]).to(DEVICE))
        for kk, v in (extra or {}).items(): b[kk] = torch.from_numpy(v[j]).to(DEVICE)
        yield b

def fit(model, train_iter, val_iter, loss_fn, ckpt_path, epochs=30, lr=1e-3, wd=1e-4,
        patience=6, verbose=True, clip=1.0):
    """Resumable training loop. `train_iter(epoch)` / `val_iter()` yield batches,
    `loss_fn(model, batch)` returns a scalar loss. A checkpoint is written after
    every epoch (model, optimiser, scheduler, history, early-stopping state);
    re-running the cell resumes where it stopped; a finished run is skipped."""
    ckpt_path = Path(ckpt_path); ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    st = dict(epoch=0, best=np.inf, bad=0, hist=[], done=False, best_state=None)
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        st = ck["state"]
        if st["done"]:
            model.load_state_dict(st["best_state"]); model.eval()
            return model, pd.DataFrame(st["hist"])
        if verbose: print(f"  resuming {ckpt_path.name} at epoch {st['epoch']}")
    for ep in range(st["epoch"], epochs):
        model.train(); t0 = time.time(); tl, nb = 0.0, 0
        for b in train_iter(ep):
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
                loss = loss_fn(model, b)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update(); tl += loss.item(); nb += 1
        sched.step()
        model.eval(); vl, vb = 0.0, 0
        with torch.no_grad():
            for b in val_iter():
                with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
                    vl += loss_fn(model, b).item(); vb += 1
        vl /= max(vb, 1); tl /= max(nb, 1)
        st["hist"].append(dict(epoch=ep, train=tl, val=vl, sec=time.time() - t0))
        if vl < st["best"] - 1e-5:
            st["best"], st["bad"] = vl, 0
            st["best_state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            st["bad"] += 1
        st["epoch"] = ep + 1
        stop = st["bad"] >= patience or st["epoch"] >= epochs
        st["done"] = stop
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), state=st), ckpt_path)
        if verbose: print(f"  ep {ep:02d} train {tl:.4f} val {vl:.4f} ({time.time()-t0:.1f}s)")
        if stop: break
    if st["best_state"] is not None: model.load_state_dict(st["best_state"])
    model.eval()
    return model, pd.DataFrame(st["hist"])

@torch.no_grad()
def predict(model, batches, key="x", fwd=None):
    model.eval(); out = []
    for b in batches:
        with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
            p = fwd(model, b) if fwd else model(b[key])
        out.append(p.float().cpu().numpy())
    return np.concatenate(out, 0)

# ---------------------------------------------------------------------------
# Building blocks shared by several papers
# ---------------------------------------------------------------------------
class PatchEncoder(nn.Module):
    """PatchTST-style temporal encoder: (B, L, C) -> (B, d)."""
    def __init__(self, L, C, d=64, patch=6, layers=2, heads=4, dropout=0.1):
        super().__init__()
        assert L % patch == 0
        self.patch, self.np = patch, L // patch
        self.proj = nn.Linear(patch * C, d)
        self.pos = nn.Parameter(torch.randn(1, self.np, d) * 0.02)
        el = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(el, layers, enable_nested_tensor=False)
        self.out = nn.Sequential(nn.LayerNorm(2 * d), nn.Linear(2 * d, d))
    def tokens(self, x):
        B, L, C = x.shape
        z = x.reshape(B, self.np, self.patch * C)
        return self.enc(self.proj(z) + self.pos)                 # (B, np, d)
    def forward(self, x):
        h = self.tokens(x)
        return self.out(torch.cat([h.mean(1), h[:, -1]], -1))

class GRUEncoder(nn.Module):
    def __init__(self, C, d=64, layers=1):
        super().__init__(); self.rnn = nn.GRU(C, d, layers, batch_first=True)
    def forward(self, x): return self.rnn(x)[0][:, -1]

class LSTMEncoder(nn.Module):
    def __init__(self, C, d=64, layers=2, dropout=0.1):
        super().__init__(); self.rnn = nn.LSTM(C, d, layers, batch_first=True, dropout=dropout)
    def forward(self, x): return self.rnn(x)[0][:, -1]

class MLPEncoder(nn.Module):
    def __init__(self, L, C, d=64):
        super().__init__(); self.net = nn.Sequential(nn.Flatten(), nn.Linear(L * C, 2 * d), nn.GELU(), nn.Linear(2 * d, d))
    def forward(self, x): return self.net(x)

class Head(nn.Module):
    def __init__(self, d, H, dropout=0.1):
        super().__init__(); self.net = nn.Sequential(nn.LayerNorm(d), nn.Dropout(dropout), nn.Linear(d, d), nn.GELU(), nn.Linear(d, H))
    def forward(self, h): return self.net(h)

In [ ]:
# ==== 1d. BUILD / LOAD THE CLEAN PANEL (cached on Drive) ====================
prep = build_prep(DATA_PATH, PREP_DIR)
meta, FOLDS = prep["meta"], prep["folds"]
CATS = sorted(meta.category.unique()); CAT2ID = {c: i for i, c in enumerate(CATS)}
for f in FOLDS:
    d = prep["dates"]
    print(f"fold {f['fold']}: train ..{d[f['train_end']].date()} | val {d[f['val_start']].date()}..{d[f['val_end']].date()}"
          f" | test {d[f['test_start']].date()}..{d[f['test_end']].date()}")
print(meta.tier.value_counts().to_dict())

In [ ]:
# ==== 2. DATASET OVERVIEW ===================================================
dataset_overview(prep)

In [ ]:
# ==== 3. CONFIGURATION ======================================================
import copy
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
L, H = 60, H_MAX
KS = [5, 10, 20, 30, 60]
IDX = subset_idx(prep, ("T1", "T2"), smoke_n=80 if SMOKE else None)
N = len(IDX)
CAT = np.array([CAT2ID[c] for c in meta.category.to_numpy()[IDX]])
F = build_features(prep, IDX)
FOLD_IDS = [2] if SMOKE else [0, 1, 2]
SPLIT_SEEDS = [0] if SMOKE else [0, 1, 2]
EPOCHS = 2 if SMOKE else 30
PATIENCE = 5
N_NB = 5                                                     # prototype neighbours
REPTILE_ITERS = 20 if SMOKE else 400
FT_STEPS = 10 if SMOKE else 60
C_FS = N_CH + 1                                              # + 'available' channel
D = {f: fold_data(prep, F, IDX, FOLDS[f], L) for f in FOLD_IDS}
PRED_DIR = RES_DIR / "preds"; PRED_DIR.mkdir(exist_ok=True)

def product_split(seed):
    sss = StratifiedShuffleSplit(1, test_size=0.2, random_state=seed)
    lab = meta.vol_bucket.to_numpy()[IDX]
    seen, unseen = next(sss.split(np.zeros(N), lab))
    return np.sort(seen), np.sort(unseen)
SPLITS = {s: product_split(s) for s in SPLIT_SEEDS}
print(f"N={N} | seen {len(SPLITS[SPLIT_SEEDS[0]][0])} | unseen {len(SPLITS[SPLIT_SEEDS[0]][1])} | folds {FOLD_IDS}")

# name-similarity neighbours (TF-IDF char n-grams + same-category bonus); neighbours are always SEEN products
TF = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit_transform([prep["names"][i].lower() for i in IDX])
SIM = (TF @ TF.T).toarray() + 0.5 * (CAT[:, None] == CAT[None, :])   # name similarity + same-category bonus
np.fill_diagonal(SIM, -1)
def neighbours(seen):
    S = SIM[:, seen]
    nb_local = np.argsort(-S, 1)[:, :N_NB]
    nb = seen[nb_local]                                      # (N, N_NB) column indices into IDX
    w = np.take_along_axis(S, nb_local, 1); w = np.maximum(w, 0) + 1e-3
    return nb, (w / w.sum(1, keepdims=True)).astype(np.float32)

## 4. Few-shot input representation
A K-shot input keeps the usual 60-step window but blanks everything older than K days and adds an
*available* channel, so one network can read any context length. **Context-length augmentation (CLA)**
samples K at random during training (half from {5, 10, 20, 30, 60}, half uniformly in 5..60).

In [ ]:
# ==== 4. TRUNCATION + ADAPTATION SETS =======================================
def truncate(X, K):
    """X (..., L, 6) -> (..., L, 7); K scalar or array broadcastable to X.shape[:-2]."""
    K = np.asarray(K)
    Lx = X.shape[-2]
    avail = (np.arange(Lx) >= (Lx - K[..., None])) if K.ndim else (np.arange(Lx) >= Lx - K)
    avail = np.broadcast_to(avail, X.shape[:-1])
    Z = np.concatenate([X, avail[..., None].astype(np.float32)], -1)
    for c in (0, 1, 2, 5): Z[..., c] = np.where(avail, Z[..., c], 0.0)
    cap = np.log1p(np.maximum(K - 1, 0)) / 5
    Z[..., 5] = np.minimum(Z[..., 5], cap[..., None] if K.ndim else cap)
    return Z.astype(np.float32)

def truncate_t(x, K):
    """torch version with a per-sample K vector (B,)."""
    Lx = x.shape[-2]
    avail = (torch.arange(Lx, device=x.device)[None] >= (Lx - K[:, None])).float()
    z = torch.cat([x, avail[..., None]], -1)
    keep = torch.ones_like(z); keep[..., [0, 1, 2, 5]] = avail[..., None].expand(-1, -1, 4)
    z = z * keep
    z[..., 5] = torch.minimum(z[..., 5], (torch.log1p((K - 1).clamp(min=0).float()) / 5)[:, None])
    return z

def adapt_set(cols, o0, K, H_=H):
    """Support windows inside the K-day context that ends at origin o0, for products `cols`.
    Sub-origins s = o0-K+1 .. o0-1; window at s may use only the (s - o0 + K) days of context
    that exist by then; targets are masked beyond o0 (no peeking at the test window)."""
    s = np.arange(o0 - K + 1, o0)
    X, Y, M, _ = gather_windows(F, IDX, s, L, H_, o0, prep["cf_day"], rows=cols)   # (len(s), n, ...)
    avail = (s - (o0 - K + 1) + 1)[:, None] * np.ones((1, len(cols)), int)
    return truncate(X, avail), Y, M

## 5. Models
A PatchTST encoder (7 input channels), category embedding, and an MLP body ending in a linear output layer.
**Prototype conditioning (MetaProto):** the encoder also reads the full 60-day windows of the product's 5 most similar *seen* products (TF-IDF name similarity plus a same-category bonus)
at the same origin; their similarity-weighted mean representation enters
through a learned gate. **Adaptation** learns a per-product additive update of the output layer (all unseen
products at once, each with its own parameters), with an L2 penalty toward zero (L2-SP).

In [ ]:
# ==== 5. MODEL + TRAINING ===================================================
class FSModel(nn.Module):
    def __init__(self, proto=False, d=64, H=H_MAX):
        super().__init__()
        self.proto = proto
        self.enc = PatchEncoder(L, C_FS, d); self.cat = nn.Embedding(len(CATS), d)
        if proto:
            self.gate = nn.Linear(2 * d, d); self.proj = nn.Linear(d, d)
        self.body = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(), nn.Dropout(0.1))
        self.out = nn.Linear(d, H)
    def feats(self, x, cat, xnb=None, wnb=None):
        z = self.enc(x) + self.cat(cat)
        if self.proto:
            B, k = xnb.shape[:2]
            zn = self.enc(xnb.reshape(B * k, L, C_FS)).view(B, k, -1)
            zn = (wnb[..., None].to(zn.dtype) * zn).sum(1)
            z = z + torch.sigmoid(self.gate(torch.cat([z, zn], -1))) * self.proj(zn)
        return self.body(z)
    def forward(self, x, cat, xnb=None, wnb=None, delta=None):
        z = self.feats(x, cat, xnb, wnb); out = self.out(z)
        if delta is not None:
            W, b = delta; out = out + torch.einsum("bd,bdh->bh", z.float(), W).to(out.dtype) + b.to(out.dtype)
        return out

class Bank:
    """Windows for one split part on the device, addressable by (origin index, product column)."""
    def __init__(self, part):
        X, Y, M, meta_ = part
        self.X, self.Y, self.M = (torch.from_numpy(a).to(DEVICE) for a in (X, Y, M))
        self.k = X.shape[0]
    def batches(self, cols, bs, shuffle, rng=None, K=None, cla=False, nb=None, wnb=None):
        pairs = np.stack(np.meshgrid(np.arange(self.k), cols, indexing="ij"), -1).reshape(-1, 2)
        if shuffle: pairs = pairs[(rng or np.random.default_rng(0)).permutation(len(pairs))]
        for s in range(0, len(pairs), bs):
            p = torch.from_numpy(pairs[s: s + bs]).to(DEVICE); oi, c = p[:, 0], p[:, 1]
            B = len(p)
            if cla:
                Kv = torch.where(torch.rand(B, device=DEVICE) < 0.5,
                                 torch.as_tensor(KS, device=DEVICE)[torch.randint(len(KS), (B,), device=DEVICE)],
                                 torch.randint(5, L + 1, (B,), device=DEVICE))
            else:
                Kv = torch.full((B,), K or L, device=DEVICE)
            b = dict(x=truncate_t(self.X[oi, c], Kv), y=self.Y[oi, c], m=self.M[oi, c],
                     cat=torch.as_tensor(CAT, device=DEVICE)[c], col=c)
            if nb is not None:
                nbc = torch.as_tensor(nb, device=DEVICE)[c]                           # (B, k)
                xn = self.X[oi[:, None].expand_as(nbc), nbc]                            # (B, k, L, C)
                b["xnb"] = truncate_t(xn.reshape(-1, L, N_CH), torch.full((xn.shape[0] * xn.shape[1],), L, device=DEVICE)).view(B, -1, L, C_FS)
                b["wnb"] = torch.as_tensor(wnb, device=DEVICE)[c]
            yield b

def fwd(m, b, delta=None): return m(b["x"], b["cat"], b.get("xnb"), b.get("wnb"), delta)
def loss_fs(m, b): return masked_huber(fwd(m, b), b["y"], b["m"])

def train_global(kind, f, split_seed):
    """kind: 'full' (K=60 only) | 'cla' | 'proto' (cla + prototype)."""
    seen, _ = SPLITS[split_seed]; nb, wnb = neighbours(seen)
    key = f"{kind}__f{f}__split{split_seed}"
    set_seed(split_seed); model = FSModel(proto=(kind == "proto"))
    tr, va = Bank(D[f]["train"]), Bank(D[f]["val"]); rng = np.random.default_rng(split_seed)
    extra = dict(nb=nb, wnb=wnb) if kind == "proto" else {}
    model, hist = fit(model, lambda ep: tr.batches(seen, 256, True, rng, cla=(kind != "full"), **extra),
                      lambda: va.batches(seen, 1024, False, K=L, **extra),
                      loss_fs, CKPT_DIR / f"{key}.pt", EPOCHS, 1e-3, patience=PATIENCE, verbose=False)
    return model, (nb, wnb)

## 6. Reptile meta-learning of the output layer
Tasks are *seen* products at random training-period origins with random K. The inner loop adapts a
task-specific update of the output layer on that task's support windows (the K-day context plus the
forecast window, all inside the training period); the outer loop moves the shared output layer toward the
average adapted solution (first-order Reptile). The encoder is frozen.

In [ ]:
# ==== 6. REPTILE + FEW-SHOT ADAPTATION ======================================
def reptile(model, f, split_seed, nbw=None, tasks=32, inner=10, inner_lr=5e-3, eps=1.0):
    ck = CKPT_DIR / f"reptile_{'proto' if model.proto else 'cla'}__f{f}__split{split_seed}.pt"
    if ck.exists():
        model.out.load_state_dict(torch.load(ck, map_location=DEVICE)); return model
    seen, _ = SPLITS[split_seed]; rng = np.random.default_rng(100 + split_seed)
    te = FOLDS[f]["train_end"]; model.eval()
    for it in range(REPTILE_ITERS):
        K = int(rng.choice(KS)); cols = rng.choice(seen, tasks, replace=False)
        o0 = int(rng.integers(L + K, te - H))
        Xs, Ys, Ms = adapt_set(cols, o0 + H, K + H, H)           # context K days + H future days (all in train)
        Xs, Ys, Ms = (torch.from_numpy(np.ascontiguousarray(a.transpose(1, 0, *range(2, a.ndim)))).to(DEVICE) for a in (Xs, Ys, Ms))
        n_s = Xs.shape[1]
        cat = torch.as_tensor(CAT[cols], device=DEVICE)[:, None].expand(-1, n_s).reshape(-1)
        with torch.no_grad():
            kw = {}
            if model.proto:
                nb, wnb = nbw
                xn = gather_windows(F, IDX, np.arange(o0 - K + 1, o0 + H), L, H, o0 + H, prep["cf_day"])[0]  # (s, N, L, C)
                xn = truncate(xn, L)
                s_idx = np.arange(n_s)
                XN = torch.from_numpy(xn[s_idx[None, :, None], nb[cols][:, None, :]]).to(DEVICE)          # (tasks, s, k, L, C)
                kw = dict(xnb=XN.reshape(-1, N_NB, L, C_FS), wnb=torch.as_tensor(wnb[cols], device=DEVICE)[:, None].expand(-1, n_s, -1).reshape(-1, N_NB))
            z = model.feats(Xs.reshape(-1, L, C_FS), cat, kw.get("xnb"), kw.get("wnb")).float()
        z = z.view(tasks, n_s, -1)
        W = torch.zeros(tasks, z.shape[-1], H, device=DEVICE, requires_grad=True)
        bb = torch.zeros(tasks, H, device=DEVICE, requires_grad=True)
        base_w, base_b = model.out.weight.detach().float(), model.out.bias.detach().float()
        opt = torch.optim.Adam([W, bb], lr=inner_lr)             # Adam inner loop, as at test time
        for _ in range(inner):
            pred = z @ (base_w.T[None] + W) + (base_b + bb)[:, None]
            loss = masked_huber(pred, Ys, Ms)
            opt.zero_grad(); loss.backward(); opt.step()
        e = eps * (1 - it / REPTILE_ITERS)                       # linearly annealed outer step
        with torch.no_grad():
            model.out.weight += e * W.mean(0).T.to(model.out.weight.dtype)
            model.out.bias += e * bb.mean(0).to(model.out.bias.dtype)
    torch.save(model.out.state_dict(), ck)
    return model

def few_shot_adapt(model, cols, o0, K, nbw=None, steps=FT_STEPS, lr=5e-3, l2=1e-2):
    """Per-product additive update (W_p, b_p) of the output layer, fitted jointly for all unseen
    products on their own K-day support windows ending at o0."""
    if K < 2: return None
    Xs, Ys, Ms = adapt_set(cols, o0, K)
    Xs, Ys, Ms = (torch.from_numpy(np.ascontiguousarray(a.transpose(1, 0, *range(2, a.ndim)))).to(DEVICE) for a in (Xs, Ys, Ms))
    P, n_s = Xs.shape[:2]
    cat = torch.as_tensor(CAT[cols], device=DEVICE)[:, None].expand(-1, n_s).reshape(-1)
    model.eval()
    with torch.no_grad():
        kw = {}
        if model.proto:
            nb, wnb = nbw
            xn = truncate(gather_windows(F, IDX, np.arange(o0 - K + 1, o0), L, H, o0, prep["cf_day"])[0], L)
            XN = torch.from_numpy(xn[np.arange(n_s)[None, :, None], nb[cols][:, None, :]]).to(DEVICE)
            kw = dict(xnb=XN.reshape(-1, N_NB, L, C_FS), wnb=torch.as_tensor(wnb[cols], device=DEVICE)[:, None].expand(-1, n_s, -1).reshape(-1, N_NB))
        z = model.feats(Xs.reshape(-1, L, C_FS), cat, kw.get("xnb"), kw.get("wnb")).float().view(P, n_s, -1)
        base = model.out(z.to(next(model.out.parameters()).dtype)).float()
    W = torch.zeros(P, z.shape[-1], H, device=DEVICE, requires_grad=True)
    bb = torch.zeros(P, H, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([W, bb], lr=lr)
    for _ in range(steps):
        pred = base + z @ W + bb[:, None]
        loss = masked_huber(pred, Ys, Ms) + l2 * (W.pow(2).sum() + bb.pow(2).sum()) / P
        opt.zero_grad(); loss.backward(); opt.step()
    return W.detach(), bb.detach()

@torch.no_grad()
def predict_unseen(model, f, split_seed, K, delta=None, nbw=None):
    _, unseen = SPLITS[split_seed]
    te = Bank(D[f]["test"]); out = []
    extra = dict(nb=nbw[0], wnb=nbw[1]) if model.proto else {}
    pos = {c: j for j, c in enumerate(unseen)}
    for b in te.batches(unseen, 1024, False, K=K, **extra):
        d = None
        if delta is not None:
            j = torch.as_tensor([pos[int(c)] for c in b["col"].cpu().numpy()], device=DEVICE)
            d = (delta[0][j], delta[1][j])
        with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
            out.append(fwd(model, b, d).float().cpu().numpy())
    return np.concatenate(out).reshape(D[f]["test"][0].shape[0], len(unseen), H)

## 7. Run all methods (resumable: predictions cached per split, fold, K and method)

In [ ]:
# ==== 7. EXPERIMENT LOOP ====================================================
def local_forecasts(X, K):
    v = X[..., -K:, 0]                                                    # (..., K) rel-log to origin
    hh = np.arange(1, H + 1)
    slope = (v[..., -1] - v[..., 0]) / max(K - 1, 1)
    drift = slope[..., None] * hh
    best, best_sse = None, None
    for a in np.linspace(0.05, 0.95, 19):
        lev = v[..., 0].copy(); sse = np.zeros_like(lev)
        for t in range(1, K):
            sse += (v[..., t] - lev) ** 2; lev = a * v[..., t] + (1 - a) * lev
        if best is None: best, best_sse = lev.copy(), sse
        else:
            m = sse < best_sse; best = np.where(m, lev, best); best_sse = np.where(m, sse, best_sse)
    return {"Naive": np.zeros(v.shape[:-1] + (H,), np.float32), "Local drift": drift.astype(np.float32),
            "Local SES": np.repeat(best[..., None], H, -1).astype(np.float32)}

METHODS = ["Naive", "Local drift", "Local SES", "Global (full-context)", "Global + CLA", "CLA + FT",
           "Reptile + FT", "Proto (zero-shot)", "MetaProto (proposed)", "Oracle (60-day history)"]
RESULTS = {}                                                              # (split, fold, K, method) -> pred
for sp in SPLIT_SEEDS:
    seen, unseen = SPLITS[sp]
    for f in FOLD_IDS:
        fold = FOLDS[f]; o0 = fold["test_start"] - 1
        cache = RES_DIR / f"preds_split{sp}_fold{f}.pkl"
        if cache.exists():
            RESULTS.update(pickle.load(open(cache, "rb"))); print("cached", cache.name); continue
        t0 = time.time(); out = {}
        m_full, _ = train_global("full", f, sp)
        m_cla, _ = train_global("cla", f, sp)
        m_pro, nbw = train_global("proto", f, sp)
        Xte = D[f]["test"][0][:, unseen]
        for K in KS:
            for m, p in local_forecasts(Xte, K).items(): out[(sp, f, K, m)] = p
            out[(sp, f, K, "Global (full-context)")] = predict_unseen(m_full, f, sp, K)
            out[(sp, f, K, "Global + CLA")] = predict_unseen(m_cla, f, sp, K)
            out[(sp, f, K, "CLA + FT")] = predict_unseen(m_cla, f, sp, K, few_shot_adapt(m_cla, unseen, o0, K))
            out[(sp, f, K, "Proto (zero-shot)")] = predict_unseen(m_pro, f, sp, K, nbw=nbw)
            out[(sp, f, K, "Oracle (60-day history)")] = predict_unseen(m_full, f, sp, L)
        m_cla_r = reptile(copy.deepcopy(m_cla), f, sp); m_pro_r = reptile(copy.deepcopy(m_pro), f, sp, nbw)
        for K in KS:
            out[(sp, f, K, "Reptile + FT")] = predict_unseen(m_cla_r, f, sp, K, few_shot_adapt(m_cla_r, unseen, o0, K))
            out[(sp, f, K, "MetaProto (proposed)")] = predict_unseen(m_pro_r, f, sp, K, few_shot_adapt(m_pro_r, unseen, o0, K, nbw), nbw)
        pickle.dump(out, open(cache, "wb")); RESULTS.update(out)
        print(f"split {sp} fold {f}: {time.time()-t0:.0f}s")

## 8. Results: data efficiency

In [ ]:
# ==== 8. EVALUATION TABLES ==================================================
def eval_cell(sp, f, K, m):
    _, unseen = SPLITS[sp]; te = D[f]["test"]
    Y, M, mt = te[1][:, unseen], te[2][:, unseen], te[3][:, unseen]
    s, per = evaluate(RESULTS[(sp, f, K, m)], Y, M, mt, prep, FOLDS[f])
    return s, pd.concat(per, axis=1).mean(1)

ROWS, PER = [], {}
for (sp, f, K, m) in RESULTS:
    s, per = eval_cell(sp, f, K, m)
    ROWS.append(dict(split=sp, fold=f, K=K, method=m, sMAPE=s.sMAPE.mean(), CC_sMAPE=s.CC_sMAPE.mean(),
                     WAPE=s.WAPE.mean(), MAE=s.MAE.mean()))
    PER[(sp, f, K, m)] = per
R = pd.DataFrame(ROWS)
agg = R.groupby(["K", "method"]).mean(numeric_only=True).reset_index()
for metric, cap in (("sMAPE", "sMAPE (%)"), ("CC_sMAPE", "CC-sMAPE (%) on targets where the price changes")):
    t = agg.pivot(index="K", columns="method", values=metric)[METHODS].reset_index()
    save_table(t, f"tab_fewshot_{metric}",
               f"Few-shot forecasting of unseen products: {cap} by history length K (mean over product splits "
               f"and rolling folds, horizons 1-30).", nd=3, bold_row_min=[m for m in METHODS if not m.startswith("Oracle")])

# per-product paired comparison of the proposed method vs each alternative, per K
rows = []
for K in KS:
    blk = []
    for m in METHODS:
        if m == "MetaProto (proposed)": continue
        a = pd.concat([PER[(sp, f, K, "MetaProto (proposed)")] for sp in SPLIT_SEEDS for f in FOLD_IDS]).groupby(level=0).mean()
        b = pd.concat([PER[(sp, f, K, m)] for sp in SPLIT_SEEDS for f in FOLD_IDS]).groupby(level=0).mean()
        j = a.index.intersection(b.index); mean, lo, hi = bootstrap_ci((b[j] - a[j]).values)
        blk.append(dict(K=K, vs=m, d_sMAPE=mean, CI_lo=lo, CI_hi=hi, p_Wilcoxon=wilcoxon_paired(a, b), n_products=len(j)))
    blk = pd.DataFrame(blk); blk["p_holm"] = holm(blk.p_Wilcoxon); rows.append(blk)
save_table(pd.concat(rows, ignore_index=True), "tab_fewshot_significance",
           "MetaProto vs each method: d_sMAPE = other minus MetaProto per-product sMAPE (positive favours "
           "MetaProto), bootstrap 95% CI, Wilcoxon signed-rank p (Holm-adjusted within K).", nd=4)

In [ ]:
# ==== 8.1 FIGURE: DATA-EFFICIENCY CURVES WITH 95% CI ========================
def curve(m, metric):
    g = R[R.method == m].groupby("K")[metric]
    mu = g.mean(); se = g.std().fillna(0) / np.sqrt(g.count())
    return mu, 1.96 * se
show = ["Naive", "Local SES", "Global (full-context)", "Global + CLA", "CLA + FT", "Reptile + FT", "MetaProto (proposed)"]
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.7))
for j, metric in enumerate(("sMAPE", "CC_sMAPE")):
    for c, m in enumerate(show):
        mu, ci = curve(m, metric)
        ax[j].plot(mu.index, mu.values, marker="o", color=PALETTE[c], label=m, lw=2.0 if "proposed" in m else 1.4)
        ax[j].fill_between(mu.index, mu - ci, mu + ci, color=PALETTE[c], alpha=0.12, lw=0)
    mu, _ = curve("Oracle (60-day history)", metric)
    ax[j].axhline(mu.mean(), color=GREY, ls="--", lw=1, label="Oracle (60-day history)")
    ax[j].set_xscale("log"); ax[j].set_xticks(KS, KS); ax[j].minorticks_off()
    ax[j].set_xlabel("History available K (days)"); ax[j].set_ylabel(metric.replace("_", "-") + " (%)")
    ax[j].set_title(f"({'ab'[j]}) {metric.replace('_', '-')}")
ax[1].legend(loc="center left", bbox_to_anchor=(1.0, 0.5)); fig.tight_layout(); savefig(fig, "fig_data_efficiency")

In [ ]:
# ==== 8.2 FIGURE: GAIN OVER NAIVE BY CATEGORY (K = 10) AND BY VOLATILITY ====
K0 = 10 if 10 in KS else KS[0]
def per_prod(m, K):
    return pd.concat([PER[(sp, f, K, m)] for sp in SPLIT_SEEDS for f in FOLD_IDS]).groupby(level=0).mean()
base = per_prod("Naive", K0)
rel = pd.DataFrame({m: (1 - per_prod(m, K0) / base.replace(0, np.nan)) * 100 for m in ["Global + CLA", "CLA + FT", "MetaProto (proposed)"]})
rel = rel.join(meta[["category", "vol_bucket"]])
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.8), gridspec_kw=dict(width_ratios=[2, 1]))
bc = rel.groupby("category").mean(numeric_only=True).sort_values("MetaProto (proposed)")
y = np.arange(len(bc)); hgt = 0.26
for c, m in enumerate(bc.columns):
    ax[0].barh(y + (c - 1) * hgt, bc[m], height=hgt, color=PALETTE[c + 2], label=m)
ax[0].set_yticks(y, bc.index, fontsize=6); ax[0].axvline(0, color=GREY, lw=0.8)
ax[0].set_xlabel(f"sMAPE reduction vs Naive at K={K0} (%)"); ax[0].legend(fontsize=6)
bv = rel.groupby("vol_bucket").mean(numeric_only=True).reindex(["low", "medium", "high"])
for c, m in enumerate(bv.columns):
    ax[1].bar(np.arange(3) + (c - 1) * hgt, bv[m], width=hgt, color=PALETTE[c + 2])
ax[1].set_xticks(range(3), bv.index); ax[1].axhline(0, color=GREY, lw=0.8); ax[1].set_xlabel("Training volatility")
ax[0].set_title("(a) By category"); ax[1].set_title("(b) By volatility"); fig.tight_layout(); savefig(fig, "fig_gain_by_group")

## 9. What does the prototype attend to?
Examples of unseen products and the seen products whose representations condition their forecasts.

In [ ]:
# ==== 9. PROTOTYPE NEIGHBOURS ===============================================
seen, unseen = SPLITS[SPLIT_SEEDS[0]]; nb, wnb = neighbours(seen)
rows = []
for c in unseen[np.argsort(-meta.loc[IDX[unseen], "n_changes_train"].to_numpy())[:8]]:
    rows.append(dict(unseen_product=prep["names"][IDX[c]][:40],
                     neighbours="; ".join(f"{prep['names'][IDX[j]][:26]} ({w:.2f})" for j, w in zip(nb[c][:3], wnb[c][:3]))))
save_table(pd.DataFrame(rows), "tab_prototype_examples", "Name-similar seen products used as prototypes (weights).")

In [ ]:
# ==== 10. ASSET INDEX + RESULTS SUMMARY =====================================
t = agg.pivot(index="K", columns="method", values="sMAPE")
lines = [f"# Paper 2 - results summary ({time.strftime('%Y-%m-%d')})",
         f"* {N} active food SKUs; {len(SPLIT_SEEDS)} product splits; folds {FOLD_IDS}; unseen per split ~{len(SPLITS[SPLIT_SEEDS[0]][1])}."]
for K in KS:
    r = t.loc[K].drop("Oracle (60-day history)")
    lines.append(f"* K={K}: best = {r.idxmin()} ({r.min():.3f}); Naive {r['Naive']:.3f}; MetaProto {r['MetaProto (proposed)']:.3f}")
(PAPER_DIR / "RESULTS_SUMMARY.md").write_text("\n".join(lines)); print("\n".join(lines))
print("figures:", sorted(p.name for p in FIG_DIR.glob("*.pdf"))); print("tables:", sorted(p.name for p in TAB_DIR.glob("*.tex")))